GradCam Implementation

In [ ]:
import os
import numpy as np
import pandas as pd
import cv2
import torch
import torch.nn as nn
from PIL import Image
from torchvision import transforms
import torchvision.models as models
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

# ==========================================================
# CONFIG — matched to your EfficientNet-B3 + Transformer training run
# ==========================================================

DATASET_ROOT = "/content/drive/MyDrive/MCI_dataset"
CSV_TO_EXPLAIN = os.path.join(DATASET_ROOT, "split/test.csv")   # change to val.csv/train.csv if needed

BACKBONE = "efficientnet_b3"
OUTPUT_DIR = f"/content/drive/MyDrive/MCI_Results_{BACKBONE}_TransformerConcatFusion"
CHECKPOINT_PATH = os.path.join(OUTPUT_DIR, "best_model.pth")
GRADCAM_DIR = os.path.join(OUTPUT_DIR, "gradcam")
OVERLAY_DIR = os.path.join(GRADCAM_DIR, "overlays")     # individual per-drawing overlay PNGs
os.makedirs(GRADCAM_DIR, exist_ok=True)
os.makedirs(OVERLAY_DIR, exist_ok=True)

IMAGE_SIZE = 300                  # EfficientNet-B3 native — MATCHED to training script
EMBED_DIM = 768                   # MATCHED to training script
NUM_HEADS = 8                     # MATCHED to training script
NUM_TRANSFORMER_LAYERS = 1        # MATCHED to training script
TRANSFORMER_FFN_DIM = 1536        # MATCHED to training script
TRANSFORMER_DROPOUT = 0.10        # MATCHED to training script (inactive anyway under model.eval())
ATTENTION_HIDDEN = 256            # MATCHED to training script
UNFREEZE_FRACTION = 0.30          # only affects which layers were trainable, not architecture shape
NUM_CLASSES = 2

# Which class to explain: None = model's own predicted class (recommended,
# explains "why did the model decide what it decided"). Set to 1 to always
# explain the MCI class regardless of prediction, or 0 for Normal.
TARGET_CLASS = None

MAX_PATIENTS = None   # e.g. set to 20 to only process the first 20 rows while testing
ALPHA = 0.45           # heatmap overlay opacity
HOT_THRESHOLD = 0.5   # CAM value above which a pixel counts as "hot" for hot_area_pct

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

model_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])


# ==========================================================
# MODEL DEFINITION — must match the training script exactly
# (module names, order, and forward-pass flags such as
# norm_first) so the checkpoint's state_dict lines up AND the
# forward pass actually reproduces training-time behavior.
# ==========================================================

def build_backbone(name):
    if name.lower() == "efficientnet_b3":
        weights = models.EfficientNet_B3_Weights.IMAGENET1K_V1
        backbone = models.efficientnet_b3(weights=weights)
        feature_dim = backbone.classifier[1].in_features   # 1536
        backbone.classifier = nn.Identity()
    else:
        raise ValueError("This Grad-CAM script is configured for efficientnet_b3 only.")
    return backbone, feature_dim


class AttentionPool(nn.Module):
    def __init__(self, embed_dim, hidden_dim=256):
        super().__init__()
        self.attention = nn.Sequential(
            nn.Linear(embed_dim, hidden_dim),
            nn.Tanh(),
            nn.Linear(hidden_dim, 1),
        )

    def forward(self, x):
        # x: [B, 3, D]
        scores = self.attention(x).squeeze(-1)
        weights = torch.softmax(scores, dim=1)
        pooled = torch.sum(x * weights.unsqueeze(-1), dim=1)
        return pooled, weights


class ResidualAttentionFusion(nn.Module):
    def __init__(self, embed_dim, attention_hidden=256):
        super().__init__()
        self.attention_pool = AttentionPool(embed_dim=embed_dim, hidden_dim=attention_hidden)

    def forward(self, transformer_features, original_features):
        # transformer_features: [B, 3, embed_dim]
        # original_features:    [B, 3, feature_dim]  (RAW backbone output)
        transformer_pooled, attention_weights = self.attention_pool(transformer_features)
        transformer_concat = transformer_features.flatten(1)
        original_concat = original_features.flatten(1)
        fused = torch.cat([original_concat, transformer_concat, transformer_pooled], dim=1)
        return fused, attention_weights


class MCITransformerConcatFusion(nn.Module):
    def __init__(self, backbone_name="efficientnet_b3", num_classes=2,
                 embed_dim=768, num_heads=8, num_layers=1, ffn_dim=1536,
                 transformer_dropout=0.10, attention_hidden=256,
                 unfreeze_fraction=0.30):
        super().__init__()

        self.backbone_name = backbone_name.lower()
        self.backbone, feature_dim = build_backbone(backbone_name)   # feature_dim = 1536

        for param in self.backbone.parameters():
            param.requires_grad = False

        feature_children = list(self.backbone.features.children())
        n = len(feature_children)
        start = int(n * (1 - unfreeze_fraction))
        for child in feature_children[start:]:
            for param in child.parameters():
                param.requires_grad = True

        self.project = nn.Sequential(
            nn.Linear(feature_dim, embed_dim),
            nn.GELU(),
            nn.LayerNorm(embed_dim),
            nn.Dropout(0.10),
        )

        self.modality_embedding = nn.Parameter(torch.zeros(1, 3, embed_dim))
        nn.init.normal_(self.modality_embedding, mean=0.0, std=0.02)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=embed_dim, nhead=num_heads, dim_feedforward=ffn_dim,
            dropout=transformer_dropout, activation="gelu",
            batch_first=True, norm_first=True,     # MUST match training script
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)

        self.fusion = ResidualAttentionFusion(embed_dim=embed_dim, attention_hidden=attention_hidden)

        fused_dim = (feature_dim * 3) + (embed_dim * 3) + embed_dim   # 4608 + 2304 + 768 = 7680

        self.classifier = nn.Sequential(
            nn.LayerNorm(fused_dim), nn.Dropout(0.40),
            nn.Linear(fused_dim, 1024), nn.GELU(),
            nn.Dropout(0.30),
            nn.Linear(1024, 256), nn.GELU(),
            nn.Dropout(0.20),
            nn.Linear(256, num_classes),
        )

    def forward(self, clock, copy, trail, return_attention=False):
        f_clock = self.backbone(clock)
        f_copy = self.backbone(copy)
        f_trail = self.backbone(trail)

        original_stack = torch.stack([f_clock, f_copy, f_trail], dim=1)  # [B, 3, 1536]
        x = self.project(original_stack)                                 # [B, 3, embed_dim]
        x_with_modality = x + self.modality_embedding
        transformer_out = self.transformer(x_with_modality)              # [B, 3, embed_dim]

        fused, attention_weights = self.fusion(transformer_out, original_stack)
        logits = self.classifier(fused)

        if return_attention:
            return logits, attention_weights
        return logits


model = MCITransformerConcatFusion(
    backbone_name=BACKBONE, num_classes=NUM_CLASSES, embed_dim=EMBED_DIM,
    num_heads=NUM_HEADS, num_layers=NUM_TRANSFORMER_LAYERS,
    ffn_dim=TRANSFORMER_FFN_DIM, transformer_dropout=TRANSFORMER_DROPOUT,
    attention_hidden=ATTENTION_HIDDEN, unfreeze_fraction=UNFREEZE_FRACTION,
).to(DEVICE)

state_dict = torch.load(CHECKPOINT_PATH, map_location=DEVICE)
model.load_state_dict(state_dict)
model.eval()
print("Loaded checkpoint:", CHECKPOINT_PATH)


# ==========================================================
# GRAD-CAM
# ==========================================================

class GradCAM:
    """
    Hooks the last spatial conv stage of the (shared) EfficientNet-B3
    backbone. Because the backbone is called up to 3 times per
    forward pass (once per drawing), the caller must ensure the
    "target" drawing's backbone call happens LAST among the three
    -- see get_gradcam_for_drawing() below -- so the forward hook's
    final captured activation corresponds to the target drawing,
    and the backward hook (which only fires for calls that are
    part of the autograd graph) naturally only fires for the one
    branch computed with grad enabled.
    """

    def __init__(self, model, target_layer):
        self.model = model
        self.activations = None
        self.gradients = None
        self.fwd_handle = target_layer.register_forward_hook(self._save_activation)
        self.bwd_handle = target_layer.register_full_backward_hook(self._save_gradient)

    def _save_activation(self, module, inp, output):
        self.activations = output.detach()

    def _save_gradient(self, module, grad_input, grad_output):
        self.gradients = grad_output[0].detach()

    def remove(self):
        self.fwd_handle.remove()
        self.bwd_handle.remove()

    def compute_cam(self):
        # activations/gradients: [1, C, H, W]
        weights = self.gradients.mean(dim=(2, 3), keepdim=True)          # [1, C, 1, 1]
        cam = (weights * self.activations).sum(dim=1, keepdim=True)      # [1, 1, H, W]
        cam = torch.relu(cam)
        cam = cam.squeeze(0).squeeze(0)                                  # [H, W]
        cam = cam - cam.min()
        cam = cam / (cam.max() + 1e-8)
        return cam.cpu().numpy()


# Target layer: last stage of the EfficientNet-B3 feature extractor,
# i.e. the final spatial feature map before global average pooling.
# torchvision's EfficientNet exposes this the same way ConvNeXt did,
# via `.features`, so this line is unchanged in spirit.
target_layer = model.backbone.features[-1]
gradcam = GradCAM(model, target_layer)


def get_gradcam_for_drawing(model, gradcam, clock_t, copy_t, trail_t, target_drawing, target_class=None):
    """
    Computes a Grad-CAM heatmap for `target_drawing` ('clock'/'copy'/'trail')
    using the REAL fused prediction (all 3 drawings genuinely contribute to
    the forward pass, exactly as in MCITransformerConcatFusion.forward()).
    Only the target drawing's backbone call is allowed to build a gradient
    graph; the other two are computed under torch.no_grad() so no gradient
    flows through them, and the target's call is scheduled last so the
    forward hook captures the right activation.

    Returns: cam (H,W numpy array in [0,1]), target_class (int),
             probs (numpy [num_classes]), attn_weights (numpy [3])
             — attn_weights here are the ResidualAttentionFusion's
             attention-pool weights over the transformer tokens (one
             real component of the 7680-d fused vector, not the whole
             story — see CSV notes below).
    """
    model.zero_grad(set_to_none=True)

    tensors = {"clock": clock_t, "copy": copy_t, "trail": trail_t}
    order = [d for d in ["clock", "copy", "trail"] if d != target_drawing] + [target_drawing]

    raw_features = {}    # raw backbone output, 1536-d — feeds original_stack
    proj_features = {}   # projected to embed_dim, 768-d — feeds the transformer

    for name in order:
        t = tensors[name]
        if name == target_drawing:
            raw = model.backbone(t)                 # keeps grad graph
        else:
            with torch.no_grad():
                raw = model.backbone(t)
            raw = raw.detach()
        raw_features[name] = raw
        proj_features[name] = model.project(raw.unsqueeze(1)).squeeze(1)  # project expects [B,3,feat] shape internally via LayerNorm over last dim; unsqueeze/squeeze keeps a consistent [B, embed_dim] here

    original_stack = torch.stack([raw_features["clock"], raw_features["copy"], raw_features["trail"]], dim=1)  # [1, 3, 1536]
    proj_stack = torch.stack([proj_features["clock"], proj_features["copy"], proj_features["trail"]], dim=1)   # [1, 3, 768]

    x_with_modality = proj_stack + model.modality_embedding
    transformer_out = model.transformer(x_with_modality)               # [1, 3, 768]

    fused, attn_weights = model.fusion(transformer_out, original_stack)
    logits = model.classifier(fused)

    probs = torch.softmax(logits, dim=1).detach().cpu().numpy()[0]

    if target_class is None:
        target_class_used = int(logits.argmax(dim=1).item())
    else:
        target_class_used = target_class

    score = logits[:, target_class_used].sum()
    score.backward()

    cam = gradcam.compute_cam()
    cam_resized = cv2.resize(cam, (IMAGE_SIZE, IMAGE_SIZE))

    return cam_resized, target_class_used, probs, attn_weights.detach().cpu().numpy()[0]


# ==========================================================
# CAM STATISTICS
# ==========================================================

def cam_statistics(cam, threshold=HOT_THRESHOLD):
    """
    Summarize a normalized [0,1] Grad-CAM map with a few numbers so the
    CSV can be sorted/filtered without opening every figure:
      - cam_mean       : overall activation strength
      - cam_max        : peak activation (always ~1.0 since we min-max
                         normalize per-image, kept for completeness)
      - hot_area_pct   : % of pixels above `threshold` -> how large/diffuse
                         the highlighted region is (small = focal, large = diffuse)
      - peak_row/peak_col : pixel location (row, col) of the single
                         strongest point, in resized IMAGE_SIZE coordinates
      - peak_row_pct/peak_col_pct : same peak location as a 0-1 fraction
                         of image height/width (resolution-independent,
                         useful for aggregating "where" across patients,
                         e.g. top-left vs center vs bottom-right of the page)
    """
    cam_mean = float(cam.mean())
    cam_max = float(cam.max())
    hot_area_pct = float((cam >= threshold).mean() * 100.0)

    peak_idx = np.unravel_index(np.argmax(cam), cam.shape)
    peak_row, peak_col = int(peak_idx[0]), int(peak_idx[1])
    peak_row_pct = peak_row / cam.shape[0]
    peak_col_pct = peak_col / cam.shape[1]

    return {
        "cam_mean": round(cam_mean, 4),
        "cam_max": round(cam_max, 4),
        "hot_area_pct": round(hot_area_pct, 2),
        "peak_row": peak_row,
        "peak_col": peak_col,
        "peak_row_pct": round(peak_row_pct, 3),
        "peak_col_pct": round(peak_col_pct, 3),
    }


# ==========================================================
# OVERLAY HELPER
# ==========================================================

def load_display_image(path):
    """Loads an image resized to IMAGE_SIZE as an RGB uint8 numpy array
    (for visualization, separate from the normalized tensor fed to the model)."""
    img = Image.open(path).convert("RGB").resize((IMAGE_SIZE, IMAGE_SIZE))
    return np.array(img)


def overlay_heatmap(rgb_image, cam, alpha=ALPHA):
    """rgb_image: HxWx3 uint8. cam: HxW float in [0,1]."""
    heatmap = np.uint8(255 * cam)
    heatmap = cv2.applyColorMap(heatmap, cv2.COLORMAP_JET)          # BGR
    heatmap = cv2.cvtColor(heatmap, cv2.COLOR_BGR2RGB)
    overlay = (alpha * heatmap + (1 - alpha) * rgb_image).astype(np.uint8)
    return overlay


def load_model_tensor(path):
    img = Image.open(path).convert("RGB")
    tensor = model_transform(img).unsqueeze(0).to(DEVICE)
    return tensor


# ==========================================================
# MAIN LOOP OVER CSV
# ==========================================================

def run_gradcam_on_csv(csv_path, output_dir, max_patients=None, target_class=TARGET_CLASS):

    df = pd.read_csv(csv_path, dtype={"patient_id": str})
    if max_patients is not None:
        df = df.iloc[:max_patients]

    class_names = ["Normal", "MCI"]
    records = []

    for _, row in tqdm(df.iterrows(), total=len(df), desc="Grad-CAM"):

        patient_id = row["patient_id"]
        true_label = int(row["label"])

        clock_t = load_model_tensor(row["clock_path"])
        copy_t = load_model_tensor(row["copy_path"])
        trail_t = load_model_tensor(row["trail_path"])

        clock_disp = load_display_image(row["clock_path"])
        copy_disp = load_display_image(row["copy_path"])
        trail_disp = load_display_image(row["trail_path"])

        disp_imgs = {"clock": clock_disp, "copy": copy_disp, "trail": trail_disp}
        src_paths = {"clock": row["clock_path"], "copy": row["copy_path"], "trail": row["trail_path"]}

        cams, overlays, overlay_paths, stats = {}, {}, {}, {}
        pred_class, probs, attn_weights = None, None, None

        for drawing in ["clock", "copy", "trail"]:
            cam, tclass, probs_out, attn_out = get_gradcam_for_drawing(
                model, gradcam, clock_t, copy_t, trail_t, drawing, target_class
            )
            cams[drawing] = cam
            stats[drawing] = cam_statistics(cam)
            pred_class = tclass
            probs = probs_out
            attn_weights = attn_out

            overlays[drawing] = overlay_heatmap(disp_imgs[drawing], cam)

            # save each drawing's overlay as its own PNG (not just the combined figure)
            ov_path = os.path.join(OVERLAY_DIR, f"{patient_id}_{drawing}_overlay.png")
            cv2.imwrite(ov_path, cv2.cvtColor(overlays[drawing], cv2.COLOR_RGB2BGR))
            overlay_paths[drawing] = ov_path

        # ---- save the combined figure ----
        fig, axes = plt.subplots(2, 3, figsize=(12, 8))
        drawings = ["clock", "copy", "trail"]

        for col, name in enumerate(drawings):
            axes[0, col].imshow(disp_imgs[name])
            axes[0, col].set_title(f"{name.capitalize()} (original)")
            axes[0, col].axis("off")

            axes[1, col].imshow(overlays[name])
            axes[1, col].set_title(f"{name.capitalize()} — attn-pool {attn_weights[col]:.2f}")
            axes[1, col].axis("off")

        fig.suptitle(
            f"Patient {patient_id} | True: {class_names[true_label]} | "
            f"Pred: {class_names[pred_class]} (P[MCI]={probs[1]:.3f})"
        )
        fig.tight_layout()

        fig_path = os.path.join(output_dir, f"{patient_id}_gradcam.png")
        fig.savefig(fig_path, dpi=200, bbox_inches="tight")
        plt.close(fig)

        # ---- derived fields ----
        correct = int(true_label == pred_class)
        if true_label == 1 and pred_class == 1:
            case_type = "TP"
        elif true_label == 0 and pred_class == 0:
            case_type = "TN"
        elif true_label == 0 and pred_class == 1:
            case_type = "FP"
        else:
            case_type = "FN"

        confidence = float(probs[pred_class])

        attn_dict = {"clock": float(attn_weights[0]), "copy": float(attn_weights[1]), "trail": float(attn_weights[2])}
        dominant_drawing = max(attn_dict, key=attn_dict.get)

        # normalized attention entropy: 1.0 = perfectly spread across all 3
        # drawings, 0.0 = fully concentrated on a single drawing.
        # NOTE: unlike the older attention-only-fusion model, this attention
        # is only over the attention-POOLED 768-d summary — one of three
        # components in the 7680-d fused vector (the other two are the raw
        # per-drawing EfficientNet features and the full transformer output,
        # both unweighted and fully visible to the classifier). So
        # `dominant_drawing`/`attn_entropy_norm` describe the pooled-summary
        # component only, not the full basis of the model's decision the way
        # they did in the pure attention-fusion model — treat them as one
        # interpretability signal among several (the CAMs themselves and
        # hot_area_pct are the more direct "where did it look" signal).
        eps = 1e-8
        attn_arr = np.array(list(attn_dict.values()))
        entropy = -np.sum(attn_arr * np.log(attn_arr + eps))
        attn_entropy_norm = float(entropy / np.log(3))

        record = {
            "patient_id": patient_id,

            # ground truth / prediction
            "true_label": true_label,
            "true_label_name": class_names[true_label],
            "predicted_label": pred_class,
            "predicted_label_name": class_names[pred_class],
            "correct": correct,
            "case_type": case_type,               # TP / TN / FP / FN
            "confidence": round(confidence, 4),   # prob of the predicted class
            "prob_normal": float(probs[0]),
            "prob_mci": float(probs[1]),

            # attention-pool component of the fusion (see note above)
            "attn_pool_clock": attn_dict["clock"],
            "attn_pool_copy": attn_dict["copy"],
            "attn_pool_trail": attn_dict["trail"],
            "attn_pool_dominant_drawing": dominant_drawing,
            "attn_pool_entropy_norm": round(attn_entropy_norm, 4),

            # source image paths (for traceability back to raw drawings)
            "clock_path": src_paths["clock"],
            "copy_path": src_paths["copy"],
            "trail_path": src_paths["trail"],

            # output file paths
            "gradcam_figure": fig_path,
            "clock_overlay_path": overlay_paths["clock"],
            "copy_overlay_path": overlay_paths["copy"],
            "trail_overlay_path": overlay_paths["trail"],
        }

        # per-drawing Grad-CAM statistics, flattened with a prefix
        for drawing in drawings:
            for stat_name, stat_val in stats[drawing].items():
                record[f"{drawing}_{stat_name}"] = stat_val

        records.append(record)

    results_df = pd.DataFrame(records)
    summary_path = os.path.join(output_dir, "gradcam_summary.csv")
    results_df.to_csv(summary_path, index=False)

    print("\nGrad-CAM complete.")
    print("Figures saved to:", output_dir)
    print("Per-drawing overlays saved to:", OVERLAY_DIR)
    print("Summary CSV saved to:", summary_path)
    print("\nCase type breakdown:")
    print(results_df["case_type"].value_counts())

    return results_df


if __name__ == "__main__":
    results_df = run_gradcam_on_csv(
        csv_path=CSV_TO_EXPLAIN,
        output_dir=GRADCAM_DIR,
        max_patients=MAX_PATIENTS,
        target_class=TARGET_CLASS,
    )
    print(results_df.head())

Loaded checkpoint: /content/drive/MyDrive/MCI_Results_efficientnet_b3_TransformerConcatFusion/best_model.pth


Grad-CAM:   0%|          | 0/138 [00:00<?, ?it/s]


Grad-CAM complete.
Figures saved to: /content/drive/MyDrive/MCI_Results_efficientnet_b3_TransformerConcatFusion/gradcam
Per-drawing overlays saved to: /content/drive/MyDrive/MCI_Results_efficientnet_b3_TransformerConcatFusion/gradcam/overlays
Summary CSV saved to: /content/drive/MyDrive/MCI_Results_efficientnet_b3_TransformerConcatFusion/gradcam/gradcam_summary.csv

Case type breakdown:
case_type
TN    98
FN    36
TP     4
Name: count, dtype: int64
            patient_id  true_label true_label_name  predicted_label  \
0   447807127603920815           0          Normal                0   
1  4335393936854271175           0          Normal                0   
2  5944318648660480123           1             MCI                0   
3  3591375421002955833           1             MCI                0   
4  1394566867109443649           0          Normal                0   

  predicted_label_name  correct case_type  confidence  prob_normal  prob_mci  \
0               Normal        1        

Report Generation Using LLMs

In [ ]:
!pip install -U bitsandbytes>=0.46.1

In [ ]:

import os
import re
import sys
import glob
import json
import random
import hashlib
import argparse
from pathlib import Path

import pandas as pd
import torch

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM
)

# ==========================================================
# CONFIGURATION
# ==========================================================

CSV_PATH = ""  # leave empty to auto-search Google Drive
BACKBONE = "efficientnet_b3"
RESULTS_DIR = f"/content/drive/MyDrive/MCI_Results_{BACKBONE}_TransformerConcatFusion"
OUTDIR = os.path.join(RESULTS_DIR, "gradcam", "reports1")
MODEL_NAME = "Qwen/Qwen2.5-7B-Instruct"   # <-- upgraded from 3B
USE_4BIT = True
LIMIT = None  # set to None for all patients
MAX_REGENERATION_ATTEMPTS = 2  # 1 initial try + 1 retry on missing values
RUN_LLM_JUDGE = True  # set False to skip the LLM-judge evaluation pass

# ==========================================================
# NOTEBOOK DETECTION
# ==========================================================

def running_in_notebook():
    return ("ipykernel" in sys.argv[0]) or ("IPython" in sys.modules)


# ==========================================================
# GOOGLE DRIVE
# ==========================================================

def mount_drive_if_needed():
    if os.path.exists("/content") and not os.path.exists("/content/drive/MyDrive"):
        try:
            from google.colab import drive
            print("Mounting Google Drive...")
            drive.mount("/content/drive")
        except ImportError:
            pass


# ==========================================================
# FIND CSV
# ==========================================================

def find_csv_if_missing(csv_path):
    if csv_path and os.path.exists(csv_path):
        return csv_path

    # Look under your actual results folder first (fast, unambiguous),
    # then fall back to a Drive-wide search only if that comes up empty.
    scoped_path = os.path.join(RESULTS_DIR, "gradcam", "gradcam_summary.csv")
    if os.path.exists(scoped_path):
        print(f"Found:\n{scoped_path}")
        return scoped_path

    print("Not found in the expected results folder. Searching all of Google Drive "
          "for gradcam_summary.csv ...")
    matches = glob.glob("/content/drive/**/gradcam_summary.csv", recursive=True)

    if len(matches) == 0:
        raise FileNotFoundError("gradcam_summary.csv not found.")

    print(f"Found:\n{matches[0]}")
    return matches[0]


# ==========================================================
# LOAD MODEL
# ==========================================================

def load_model(model_name=MODEL_NAME, use_4bit=USE_4BIT):
    print(f"\nLoading {model_name} ...")

    tokenizer = AutoTokenizer.from_pretrained(model_name)

    kwargs = {
        "device_map": "auto" if torch.cuda.is_available() else None,
        "torch_dtype": torch.float16 if torch.cuda.is_available() else torch.float32,
    }

    if use_4bit and torch.cuda.is_available():
        try:
            from transformers import BitsAndBytesConfig
            bnb_config = BitsAndBytesConfig(
                load_in_4bit=True,
                bnb_4bit_compute_dtype=torch.float16,
                bnb_4bit_use_double_quant=True,
                bnb_4bit_quant_type="nf4",
            )
            kwargs["quantization_config"] = bnb_config
            kwargs.pop("torch_dtype")
        except Exception:
            print("bitsandbytes not installed.\nUsing FP16 instead.")

    model = AutoModelForCausalLM.from_pretrained(model_name, **kwargs)

    if not torch.cuda.is_available():
        model = model.to("cpu")

    model.eval()

    # Some Qwen checkpoints ship generation_config.json with do_sample=False
    # baked in, which can silently override generate() kwargs. Force explicitly.
    model.generation_config.do_sample = True
    model.generation_config.temperature = 0.85
    model.generation_config.top_p = 0.92
    model.generation_config.top_k = 50

    print("Model loaded successfully.\n")
    print("Confirmed do_sample:", model.generation_config.do_sample)
    print("Confirmed temperature:", model.generation_config.temperature)

    return model, tokenizer


# ==========================================================
# GENERATE REPORT
# ==========================================================

def seed_from_patient(pid, attempt=0):
    """Deterministic per-patient seed so re-runs are reproducible
    but different patients (and retry attempts) still get different
    sampling trajectories."""
    key = f"{pid}_{attempt}"
    return int(hashlib.md5(key.encode()).hexdigest(), 16) % (2**32)


def generate(
    model,
    tokenizer,
    system_prompt,
    user_prompt,
    patient_id=None,
    attempt=0,
    max_new_tokens=700,
    temperature=0.85,
):
    if patient_id is not None:
        torch.manual_seed(seed_from_patient(patient_id, attempt))

    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt},
    ]

    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer([prompt], return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=temperature,
            do_sample=True,
            top_p=0.92,
            top_k=50,
            repetition_penalty=1.15,
            pad_token_id=tokenizer.eos_token_id,
        )

    generated = outputs[0][inputs["input_ids"].shape[1]:]
    response = tokenizer.decode(generated, skip_special_tokens=True)
    return response.strip()


# ==========================================================
# STYLE VARIATION INJECTION (reduces cross-report repetition)
# ==========================================================
# The system prompt offers the model a menu of opening phrases / verbs and
# asks it to "rotate naturally" -- in practice, models given a long list of
# options tend to default to the same 1-2 choices across many calls. Instead
# of relying on the model to self-diversify, we deterministically pick a
# specific opening/verb combination per patient (seeded by patient ID, so
# results are reproducible) and instruct the model to use only that
# combination for this specific report.

SECTION1_OPENINGS = [
    "Across the submitted drawings...",
    "Comparison of the three drawing tasks...",
    "Assessment of the cognitive drawings...",
    "Evaluation of all three tasks...",
    "The integrated drawing assessment...",
    "Quantitative analysis demonstrated...",
]

SECTION2_OPENINGS = [
    "Across the submitted drawings...",
    "Evaluation of the three drawing tasks...",
    "Comparison of the quantitative observations...",
    "Assessment of the drawing tasks...",
    "The integrated drawing assessment...",
    "Quantitative analysis demonstrated...",
]

RECOMMENDATION_STYLES = [
    "Routine clinical follow-up may be considered should cognitive concerns arise.",
    "Further assessment may be appropriate if cognitive symptoms progress.",
    "Clinical reassessment can be undertaken if future concerns emerge.",
    "Ongoing observation may assist in monitoring cognitive status over time.",
]

VERB_POOL = [
    "showed", "revealed", "represented", "accounted for", "displayed",
    "illustrated", "identified", "highlighted", "demonstrated", "reflected",
    "described", "presented",
]


def pick_style(patient_id):
    """Deterministic (per-patient) style pick, independent of generation
    seed so it stays stable even if MAX_REGENERATION_ATTEMPTS changes."""
    rnd = random.Random(seed_from_patient(f"style_{patient_id}", attempt=0))
    return {
        "s1_opening": rnd.choice(SECTION1_OPENINGS),
        "s2_opening": rnd.choice(SECTION2_OPENINGS),
        "recommendation_style": rnd.choice(RECOMMENDATION_STYLES),
        "verbs": rnd.sample(VERB_POOL, 4),
    }


def build_style_block(style):
    return f"""
=========================================================
STYLE INSTRUCTIONS FOR THIS SPECIFIC REPORT (do not deviate)
=========================================================

Begin Section 1 (Cognitive Drawing Assessment) with a natural variation of:
"{style['s1_opening']}"

Begin Section 2 (Clinical Findings) with a natural variation of:
"{style['s2_opening']}"

In this report, prefer these verbs over others when describing observations:
{", ".join(style['verbs'])}

For Section 4 (Recommendation), write a natural variation of the idea:
"{style['recommendation_style']}"

Do not reuse the exact wording from other reports you may have generated
previously. This report must read as independently written.
"""


# ==========================================================
# SYSTEM PROMPT
# ==========================================================

SYSTEM_PROMPT = """
You are an experienced consultant neurologist specializing in cognitive disorders and memory clinic assessments.

Your responsibility is to prepare a professional Cognitive Drawing Assessment Report using ONLY the verified information provided by the user.

The report should resemble documentation written by an experienced neurologist after reviewing standardized cognitive drawing assessments.

==================================================
ROLE
==================================================

Write as a clinician—not as an AI assistant.

Use concise, professional medical English.

Maintain an objective and cautious clinical tone.

The report should be informative without sounding overly technical or repetitive.

Never exaggerate findings.

Never speculate.

Never diagnose diseases beyond the supplied information.

Never infer cognitive deficits that are not explicitly supported by the provided data.

All interpretations must remain conservative and clinically appropriate.

==================================================
WRITING STYLE
==================================================

Every report should read as an independently written clinical note.

Avoid producing reports that appear identical except for numerical values.

Vary naturally across reports:

• sentence length

• sentence order

• paragraph structure

• transition words

• clinical vocabulary

• opening sentences

• concluding sentences

Do NOT use identical wording across different patients.

The report should never appear copy-pasted.

Use natural clinical language.

Instead of repeatedly using the same verb, rotate naturally among expressions such as

showed

revealed

represented

accounted for

displayed

illustrated

identified

highlighted

demonstrated

reflected

described

presented

Do not intentionally force these words.

Choose whichever sounds most natural.

Avoid repeatedly beginning paragraphs with phrases such as

"The Clock Drawing..."

"Review of..."

"Collectively..."

"The overall drawing pattern..."

"The combined assessment..."

Instead vary paragraph openings naturally.

Examples include

Across the submitted drawings...

Comparison of the three drawing tasks...

Assessment of the cognitive drawings...

Evaluation of all three tasks...

The integrated drawing assessment...

Quantitative analysis demonstrated...

No two reports should begin in exactly the same way.

If the user prompt includes a "STYLE INSTRUCTIONS FOR THIS SPECIFIC REPORT"
section, you MUST follow those exact opening/verb/recommendation choices for
this report instead of freely picking from the general lists above.

==================================================
FACTUAL ACCURACY
==================================================

Only use information supplied by the user.

Never modify

• drawing names

• percentages

• probabilities

• clinical conclusion

Never calculate new values.

Never invent observations.

Never omit supplied numerical values.

Every supplied value must appear exactly once unless specifically stated otherwise.

Accuracy is more important than writing style.
==================================================
FORBIDDEN CONTENT
==================================================

The report is based only on quantitative drawing observations.

Do NOT introduce concepts that are not explicitly provided.

Never describe the results using terms that imply scoring, grading, severity, or cognitive performance.

Avoid the following words or phrases unless they are explicitly supplied by the user.

score

overall score

test score

performance

task performance

high performance

low performance

better performance

worse performance

accuracy

confidence

confidence score

severity

severity score

degree of impairment

risk score

normal range

abnormal range

within normal limits

outside normal limits

higher involvement

lower involvement

greater impairment

less impairment

clear-cut deficit

cognitive deficit

executive deficit

memory deficit

functional impairment

task difficulty

failed

failure

successful

unsuccessful

improved

declined

deterioration

==================================================
MEDICAL RESTRICTIONS
==================================================

Unless explicitly supplied, never mention or diagnose

Alzheimer's disease

Parkinson's disease

Dementia

Stroke

Executive dysfunction

Constructional apraxia

Visuospatial impairment

Language impairment

Memory impairment

Attention deficit

Planning deficit

Sequencing deficit

Motor impairment

Depression

Delirium

Neurological disorder

==================================================
TECHNICAL RESTRICTIONS
==================================================

Never mention

Artificial Intelligence

AI

Machine Learning

Deep Learning

Transformer

Neural Network

Grad-CAM

Heatmap

Activation Map

Attention Map

Vision Model

Algorithm

Prediction

Classification

Confidence

Probability Model

Feature Extraction

Saliency

CNN

Vision Transformer

==================================================
INTERPRETATION RULES
==================================================

Contribution percentages represent only the relative contribution of each drawing within the integrated assessment.

Localized atypical region percentages represent only the proportion of localized atypical regions identified within each drawing.

Neither measurement should be interpreted as

• disease severity

• cognitive performance

• intelligence

• functional ability

• impairment level

• diagnostic certainty

Do not compare drawings using terms such as

better

worse

stronger

weaker

higher performing

lower performing

more impaired

less impaired

Instead, describe the quantitative observations objectively.

==================================================
LANGUAGE STYLE
==================================================

Prefer neutral expressions such as

represented

accounted for

showed

displayed

demonstrated

revealed

included

contained

identified

illustrated

Avoid dramatic wording such as

markedly abnormal

severely impaired

significant deficit

highly concerning

strong evidence

clear evidence

definitive evidence

diagnostic of

consistent with disease

Instead use cautious medical language such as

suggests

may indicate

appears consistent with

should be interpreted in context

requires clinical correlation

may warrant follow-up

does not independently establish clinical impairment

==================================================
OUTPUT
==================================================

Generate only the completed report.

Do not include explanations.

Do not include notes.

Do not include reasoning.

Do not include placeholders.

Do not include markdown tables.

Do not include bullet lists.

Do not include numbered lists.

Write natural clinical paragraphs only.
==================================================
SECTION 1
## Cognitive Drawing Assessment
==================================================

Write ONE concise paragraph (approximately 60–80 words).

This section should objectively summarize the quantitative observations from the three drawing tasks.

Mention each drawing exactly once.

For each drawing include

• drawing name

• contribution percentage

• localized atypical region percentage

Do NOT describe the drawings in a fixed sequence.

The order may vary naturally.

For example, the paragraph may begin with

• the drawing with the greatest contribution

• the drawing with the largest localized atypical region

• an overall summary of all three drawings

• a comparison of the three drawing tasks

Choose whichever produces the most natural paragraph.

Avoid writing one sentence for every drawing.

Instead, integrate multiple observations into fluent clinical prose.

At the end of the paragraph briefly identify

• the drawing contributing the greatest proportion

• the drawing showing the largest localized atypical region

Do not interpret these observations as indicating better or worse cognitive performance.

Contribution percentages describe only the relative contribution of each drawing within the integrated assessment.

Localized atypical region percentages describe only the observed localized atypical regions.

Do not use words such as

score

performance

better

worse

normal

abnormal

severity

deficit

==================================================
SECTION 2
## Clinical Findings
==================================================

Write ONE concise paragraph consisting of two or three sentences.

This section should summarize the overall quantitative observations rather than repeating the numerical values.

Do NOT repeat percentages already presented in Section 1.

Explain that the three drawing tasks demonstrate quantitative variation and should be interpreted together.

State that individual drawings alone are insufficient for clinical interpretation.

Do not infer neurological deficits.

Do not describe any drawing as impaired, abnormal, better, worse, stronger, weaker, or more severe.

Avoid statements such as

"normal performance"

"minimal impairment"

"clear cognitive deficit"

"within normal range"

"outside normal range"

Instead, use cautious language such as

"The observations demonstrate modest quantitative variation across the three drawing tasks."

"The findings should be interpreted collectively rather than on the basis of any individual drawing."

"The quantitative observations alone do not establish clinically significant cognitive impairment."

"The integrated pattern provides objective information that should be considered alongside the broader clinical evaluation."

Every report should use different sentence structure.

Avoid repeating identical opening sentences between reports.

Possible opening styles include

Across the submitted drawings...

Evaluation of the three drawing tasks...

Comparison of the quantitative observations...

Assessment of the drawing tasks...

The integrated drawing assessment...

Quantitative analysis demonstrated...

Rotate naturally.

Do not use the same opening in consecutive reports.
==================================================
SECTION 3
## Clinical Impression
==================================================

This section should contain exactly THREE concise sentences.

---------------------------------------------
Sentence 1
---------------------------------------------

The FIRST sentence MUST reproduce the supplied clinical conclusion EXACTLY as provided.

The conclusion will be one of the following:

"The overall drawing pattern is suggestive of Mild Cognitive Impairment (MCI)."

OR

"The overall drawing pattern is not suggestive of Mild Cognitive Impairment (MCI)."

Do NOT modify this sentence.

---------------------------------------------
Sentence 2
---------------------------------------------

Report the supplied probability values exactly.

Include

• Normal cognition probability

• Mild Cognitive Impairment (MCI) probability

The numerical values must remain unchanged.

Do NOT calculate new values.

Do NOT describe one probability as high, low, significant, borderline, reassuring, concerning, or abnormal.

Present the probabilities objectively.

Examples of acceptable wording include

"The integrated assessment estimated probabilities of ..."

"The combined evaluation indicated probabilities of ..."

"Estimated probabilities were ..."

Rotate naturally between reports.

---------------------------------------------
Sentence 3
---------------------------------------------

Explain that the interpretation is based on the combined evaluation of all three drawing tasks.

Do NOT use exactly the same wording every time.

Possible natural variations include

"The interpretation reflects the integrated assessment of all three drawing tasks rather than any individual drawing."

"This conclusion is derived from the combined evaluation of the three drawing tasks."

"The overall interpretation considers the collective findings across all drawing tasks."

"The assessment integrates observations from all three drawings instead of relying on any single task."

Rotate naturally.

Avoid identical wording across reports.

==================================================
SECTION 4
## Recommendation
==================================================

Write ONE concise paragraph consisting of one or two sentences.

Recommendations should remain cautious.

Never sound overly prescriptive.

Recommendations may include ideas such as

• routine clinical follow-up

• monitoring if concerns develop

• reassessment if symptoms evolve

• formal neuropsychological evaluation when clinically indicated

Do NOT repeat the same recommendation wording across reports.

Examples of acceptable styles include

"Routine clinical follow-up may be considered should cognitive concerns arise."

"Further assessment may be appropriate if cognitive symptoms progress."

"Clinical reassessment can be undertaken if future concerns emerge."

"Ongoing observation may assist in monitoring cognitive status over time."

Rotate naturally.

Do not always begin with

"Given the current findings..."

or

"Ongoing monitoring..."

Use different sentence openings.

Never recommend treatment.

Never recommend medication.

Never recommend imaging.

Never state that the patient definitely has or does not have a neurological disorder.

Recommendations should always remain conditional.

==================================================
FINAL DISCLAIMER
==================================================

End EVERY report with EXACTLY the following sentence.

These findings should be interpreted together with the patient's clinical history, neurological examination, and formal cognitive assessment, and should not be considered diagnostic based solely on the drawing tasks.

Do NOT modify this sentence under any circumstances.
==================================================
FINAL VALIDATION
==================================================

Before returning the report, verify the following.

✓ The report contains exactly four sections

## Cognitive Drawing Assessment

## Clinical Findings

## Clinical Impression

## Recommendation

✓ Every supplied drawing name appears exactly once.

✓ Every supplied contribution percentage appears exactly once.

✓ Every supplied localized atypical region percentage appears exactly once.

✓ Both supplied probability values appear exactly once.

✓ The supplied clinical conclusion is unchanged.

✓ The mandatory disclaimer is unchanged.

✓ No numerical value has been modified.

✓ No unsupported neurological interpretation has been introduced.

✓ No forbidden terminology has been used.

✓ The report is concise, clinically appropriate and naturally written.

==================================================
WRITING QUALITY
==================================================

The report should resemble documentation written by an experienced neurologist.

Prioritize readability over complexity.

Avoid unnecessary repetition.

Avoid repeating the same sentence structure across reports.

Avoid repeatedly using the same verbs.

Avoid repeatedly using the same paragraph openings.

Avoid repetitive transitions such as

Collectively...

Overall...

Furthermore...

Moreover...

Instead, vary transitions naturally.

If two reports contain similar numerical values, their wording should still differ naturally.

Write as though every report is being prepared independently for a different patient.

Do not sound like a report template.

==================================================
OUTPUT REQUIREMENTS
==================================================

Return ONLY the completed report.

Do not explain your reasoning.

Do not describe your instructions.

Do not include notes.

Do not include placeholders.

Do not include markdown tables.

Do not include bullet points.

Do not include numbered lists.

The complete report should normally contain approximately 140–180 words.

Conciseness and clarity are preferred over long descriptions.
"""
# ==========================================================
# BUILD USER PROMPT
# ==========================================================

def build_user_prompt(row, correction_note="", style=None):

    # NOTE: these are the attention-POOL weights from
    # ResidualAttentionFusion.attention_pool (see gradcam_summary.csv,
    # columns attn_pool_clock / attn_pool_copy / attn_pool_trail). In
    # your current EfficientNet-B3 + Transformer + full-feature-concat
    # model, this is one of three components concatenated into the
    # classifier's input (alongside raw per-drawing features and the
    # full transformer output), not the sole channel through which a
    # drawing reaches the decision. The report still presents these as
    # "Contribution" percentages for readability, per the system prompt.
    attn = {
        "Clock Drawing": row["attn_pool_clock"] * 100,
        "Copy Drawing": row["attn_pool_copy"] * 100,
        "Trail Making": row["attn_pool_trail"] * 100,
    }

    hot = {
        "Clock Drawing": row["clock_hot_area_pct"],
        "Copy Drawing": row["copy_hot_area_pct"],
        "Trail Making": row["trail_hot_area_pct"],
    }

    dominant_attn = max(attn, key=attn.get)
    dominant_hot = max(hot, key=hot.get)

    prob_normal = row["prob_normal"] * 100
    prob_mci = row["prob_mci"] * 100

    conclusion = (
        "The overall drawing pattern is suggestive of Mild Cognitive Impairment (MCI)."
        if prob_mci >= 50
        else "The overall drawing pattern is not suggestive of Mild Cognitive Impairment (MCI)."
    )

    correction_block = ""

    if correction_note:
        correction_block = f"""

IMPORTANT

Your previous report omitted one or more required numerical values.

Rewrite the entire report.

Make sure every percentage listed below appears exactly once.

Missing values:
{correction_note}

"""

    style_block = build_style_block(style) if style else ""

    return f"""
Patient ID:
{row['patient_id']}

Verified clinical information for one patient is provided below.
Generate the report using this information only.

{correction_block}
{style_block}

=========================================================
VERIFIED FINDINGS
=========================================================

Drawing Contributions

Clock Drawing
Contribution: {attn['Clock Drawing']:.1f}%

Copy Drawing
Contribution: {attn['Copy Drawing']:.1f}%

Trail Making
Contribution: {attn['Trail Making']:.1f}%

---------------------------------------------------------

Localized Atypical Regions

Clock Drawing
Localized atypical region: {hot['Clock Drawing']:.1f}%

Copy Drawing
Localized atypical region: {hot['Copy Drawing']:.1f}%

Trail Making
Localized atypical region: {hot['Trail Making']:.1f}%

---------------------------------------------------------

Highest contribution

{dominant_attn}

Largest localized atypical region

{dominant_hot}

---------------------------------------------------------

Assessment Probabilities

Normal cognition:
{prob_normal:.1f}%

Mild Cognitive Impairment (MCI):
{prob_mci:.1f}%

---------------------------------------------------------

Clinical Conclusion

{conclusion}

"""

# ==========================================================
# UTILITIES
# ==========================================================

def sanitize_filename(name):
    return re.sub(r"[^A-Za-z0-9_\-]", "_", str(name))


def normalize_quotes(text):
    """Collapse curly/smart quotes (and a couple of related typographic
    substitutions) to their plain ASCII equivalents.

    Qwen2.5, like most instruction-tuned models, has a strong prior toward
    typographic punctuation and will frequently render a straight apostrophe
    (') as a curly apostrophe (') even when explicitly instructed to
    reproduce a sentence "verbatim" / "character-for-character". This is
    invisible to a human reader but breaks any exact substring match against
    a reference string written with straight quotes (e.g. DISCLAIMER_TEXT
    below, which checks for "patient's" with a straight apostrophe).

    Apply this before any exact/near-exact text-matching check that compares
    model output against a fixed reference string.
    """
    replacements = {
        "\u2018": "'", "\u2019": "'",   # left/right single curly quotes
        "\u201c": '"', "\u201d": '"',   # left/right double curly quotes
        "\u2013": "-", "\u2014": "-",   # en dash, em dash
    }
    for bad, good in replacements.items():
        text = text.replace(bad, good)
    return text


# ==========================================================
# SAFETY NET: ENFORCE CLINICAL IMPRESSION
# ==========================================================
# The Clinical Impression section states the actual MCI conclusion, which is
# the single most important clinical statement in the whole report. LLMs
# (even at low temperature) can occasionally invert short dictated phrases
# such as "suggestive of MCI" vs "not suggestive of MCI" during sampling.
# Rather than trusting the model to reproduce this correctly every time, we
# FORCE the section to the verified, programmatically-computed text after
# generation. This guarantees the conclusion can never be wrong, regardless
# of what the model generated.
#
# IMPORTANT: this also means the MCI / not-MCI wording in every report is
# determined ENTIRELY by row["prob_mci"] in your input CSV. The LLM has no
# influence over this sentence at all. If every report in your output says
# "not suggestive of MCI", check the prob_mci column of gradcam_summary.csv
# -- see print_conclusion_distribution() below, which is called
# automatically at the start of run().

def enforce_clinical_impression(report_text, row):
    prob_normal = row["prob_normal"] * 100
    prob_mci = row["prob_mci"] * 100

    conclusion = (
        "suggestive of Mild Cognitive Impairment (MCI)"
        if prob_mci >= 50
        else "not suggestive of Mild Cognitive Impairment (MCI)"
    )

    correct_block = (
        f"## Clinical Impression\n"
        f"The overall drawing pattern is {conclusion}. "
        f"The combined assessment demonstrated an estimated probability of "
        f"{prob_normal:.1f}% for Normal cognition and {prob_mci:.1f}% for "
        f"Mild Cognitive Impairment (MCI). This interpretation is based on "
        f"the integrated evaluation of all three drawing tasks rather than "
        f"any individual drawing."
    )

    pattern = r"## Clinical Impression.*?(?=\n## |\Z)"
    if re.search(pattern, report_text, flags=re.DOTALL):
        report_text = re.sub(pattern, correct_block + "\n\n", report_text, flags=re.DOTALL)
    else:
        # Section missing entirely — insert before Recommendation if present,
        # otherwise append at the end.
        if "## Recommendation" in report_text:
            report_text = report_text.replace(
                "## Recommendation", correct_block + "\n\n## Recommendation"
            )
        else:
            report_text += "\n\n" + correct_block

    return report_text


def print_conclusion_distribution(df):
    """Diagnostic: shows how many patients are MCI vs not-MCI *before* any
    generation happens. Since enforce_clinical_impression() force-overwrites
    Section 3 from row["prob_mci"], this distribution is exactly what your
    final reports will show. If this prints e.g. "Not MCI: 40, MCI: 0", the
    LLM is not the cause -- your upstream classifier / gradcam_summary.csv
    is producing prob_mci < 50% for every patient."""
    is_mci = df["prob_mci"] * 100 >= 50
    n_mci = int(is_mci.sum())
    n_not_mci = int((~is_mci).sum())
    print("=" * 70)
    print("CONCLUSION DISTRIBUTION (derived directly from CSV prob_mci column)")
    print("=" * 70)
    print(f"  Suggestive of MCI      : {n_mci}")
    print(f"  Not suggestive of MCI  : {n_not_mci}")
    if n_mci == 0 or n_not_mci == 0:
        print("  NOTE: All patients fall into a single category. This is NOT an")
        print("  LLM report-writing issue -- the conclusion sentence is force-")
        print("  overwritten from prob_mci in the CSV (see enforce_clinical_")
        print("  impression()). Check the upstream model/pipeline that produced")
        print("  prob_normal / prob_mci in gradcam_summary.csv.")
    print("=" * 70 + "\n")


# ==========================================================
# SAFETY NET: ENFORCE FINAL DISCLAIMER
# ==========================================================
# FIX: The final disclaimer must appear verbatim, but Qwen2.5 was found to
# reliably swap the straight apostrophe in "patient's" for a curly/typographic
# apostrophe even under an explicit "verbatim" instruction (visible in real
# output, e.g. "patient's cognitive status" rendered with a curly quote). That
# single-character substitution is invisible to a human reader but caused the
# exact-substring check in disclaimer_present_score() to fail, tanking the
# disclaimer_present metric even though every report actually ended with the
# correct disclaimer text. Rather than relying on the model to reproduce a
# fixed sentence with byte-for-byte punctuation fidelity, we force-overwrite
# the disclaimer after generation, exactly as already done for the Clinical
# Impression section above.

DISCLAIMER_SENTENCE = (
    "These findings should be interpreted together with the patient's "
    "clinical history, neurological examination, and formal cognitive "
    "assessment, and should not be considered diagnostic based solely "
    "on the drawing tasks."
)


def enforce_disclaimer(report_text):
    # Strip whatever disclaimer-like sentence the model produced at the very
    # end of the report (it should be the last thing generated), then append
    # the verified, exact disclaimer text.
    pattern = r"These findings should be interpreted.*$"
    stripped = re.sub(pattern, "", report_text, flags=re.DOTALL | re.IGNORECASE).rstrip()

    if stripped == report_text.rstrip():
        # No disclaimer-like sentence was found at all (model omitted it
        # entirely) — just append to whatever content exists.
        stripped = report_text.rstrip()

    return stripped + "\n\n" + DISCLAIMER_SENTENCE


def validate_conclusion_match(report_text, row, patient_id):
    """Post-enforcement sanity check — should always pass now, but logs
    loudly if something unexpected happened (e.g. regex failed to match)."""
    prob_mci = row["prob_mci"] * 100
    expected = (
        "suggestive of mild cognitive impairment (mci)."
        if prob_mci >= 50
        else "not suggestive of mild cognitive impairment (mci)."
    )
    if expected not in report_text.lower():
        print(f"CRITICAL WARNING: Patient {patient_id} — conclusion mismatch "
              f"even after enforcement. Manual review required.")
        return False
    return True


def validate_disclaimer_match(report_text, patient_id):
    """Post-enforcement sanity check for the disclaimer, mirroring
    validate_conclusion_match. Should always pass now that the disclaimer is
    force-overwritten, but logs loudly if something unexpected happened."""
    if DISCLAIMER_SENTENCE.lower() not in normalize_quotes(report_text.lower()):
        print(f"CRITICAL WARNING: Patient {patient_id} — disclaimer mismatch "
              f"even after enforcement. Manual review required.")
        return False
    return True


def get_required_numbers(row):
    attn = {
        "Clock Drawing": row["attn_pool_clock"] * 100,
        "Copy Drawing": row["attn_pool_copy"] * 100,
        "Trail Making": row["attn_pool_trail"] * 100,
    }
    hot = {
        "Clock Drawing": row["clock_hot_area_pct"],
        "Copy Drawing": row["copy_hot_area_pct"],
        "Trail Making": row["trail_hot_area_pct"],
    }
    return (
        [f"{v:.1f}" for v in attn.values()] + [f"{v:.1f}" for v in hot.values()]
    )


def check_missing_percentages(report_text, row, patient_id, verbose=True):
    """Returns a list of required percentage strings missing from the
    report body. Used both for logging and for triggering a retry."""
    required_numbers = get_required_numbers(row)
    missing = [num for num in required_numbers if num not in report_text]
    if missing and verbose:
        print(f"WARNING: Patient {patient_id} — missing percentages in report: {missing}")
    return missing


# ==========================================================
# GENERATE WITH RETRY ON MISSING VALUES
# ==========================================================

def generate_report_with_retry(
    model,
    tokenizer,
    row,
    patient_id,
    style,
    max_attempts=MAX_REGENERATION_ATTEMPTS,
):
    """Generates the report, and if any required percentages are missing,
    regenerates once with an explicit correction note listing exactly what
    was missing. Falls back to the best attempt (fewest missing values) if
    still incomplete after max_attempts."""

    best_report = None
    best_missing = None
    correction_note = ""

    for attempt in range(max_attempts):
        user_prompt = build_user_prompt(row, correction_note=correction_note, style=style)

        report = generate(
            model=model,
            tokenizer=tokenizer,
            system_prompt=SYSTEM_PROMPT,
            user_prompt=user_prompt,
            patient_id=patient_id,
            attempt=attempt,
        )

        missing = check_missing_percentages(report, row, patient_id, verbose=False)

        if best_missing is None or len(missing) < len(best_missing):
            best_report = report
            best_missing = missing

        if not missing:
            if attempt > 0:
                print(f"  -> Retry succeeded on attempt {attempt + 1} for Patient {patient_id}.")
            return report, []

        if attempt < max_attempts - 1:
            print(f"  -> Attempt {attempt + 1} for Patient {patient_id} missing values "
                  f"{missing}. Retrying with correction note...")
            correction_note = (
                f"Your previous attempt was missing the following required percentage "
                f"value(s): {', '.join(missing)}. Rewrite the report from scratch, making "
                f"sure every one of these exact values appears in Section 1 as part of its "
                f"required per-drawing sentence."
            )

    print(f"  -> Patient {patient_id}: still missing {best_missing} after {max_attempts} "
          f"attempts. Using best available attempt.")
    return best_report, best_missing


# ==========================================================
# HUMAN EVALUATION TEMPLATE
# ==========================================================

def create_human_evaluation_template(outdir, df):
    evaluation_file = Path(outdir) / "human_evaluation_template.csv"
    records = []

    for _, row in df.iterrows():
        pid = sanitize_filename(row["patient_id"])
        records.append({
            "Patient_ID": row["patient_id"],
            "Report_File": f"{pid}_report.md",
            "Clinical_correctness_1_5": "",
            "Completeness_1_5": "",
            "Readability_1_5": "",
            "Professional_medical_style_1_5": "",
            "Consistency_with_assessment_findings_1_5": "",
            "Hallucination_absence_1_5": "",
            "Overall_score_1_5": "",
            "Reviewer_comments": ""
        })

    pd.DataFrame(records).to_csv(evaluation_file, index=False)
    print(f"Human evaluation template saved:\n{evaluation_file}")


# ==========================================================
# AUTOMATED (HEURISTIC) EVALUATION MODULE
# ==========================================================

REQUIRED_SECTIONS = [
    "## Cognitive Drawing Assessment",
    "## Clinical Findings",
    "## Clinical Impression",
    "## Recommendation"
]

FORBIDDEN_TERMS = [
    "artificial intelligence", " ai ", "machine learning", "deep learning",
    "transformer", "convnext", "efficientnet", "grad-cam", "gradcam", "heatmap",
    "attention", "activation", "model", "algorithm", "prediction",
    "classification", "confidence score"
]

# NOTE: kept as straight ASCII quotes deliberately. Scoring functions that
# compare against this must run normalize_quotes() on the model output first
# (see disclaimer_present_score below) rather than "fixing" this constant,
# since the constant is the source of truth for what the enforced disclaimer
# text actually is.
DISCLAIMER_TEXT = (
    "these findings should be interpreted together with the patient's "
    "clinical history, neurological examination, and formal cognitive "
    "assessment, and should not be considered diagnostic based solely on "
    "the drawing tasks"
)


def completeness_score(report_text):
    hits = sum(1 for sec in REQUIRED_SECTIONS if sec.lower() in report_text.lower())
    return hits / len(REQUIRED_SECTIONS)


def terminology_leak_score(report_text):
    text_lower = f" {report_text.lower()} "
    leaks = [term for term in FORBIDDEN_TERMS if term.strip() in text_lower]
    score = 1.0 - (len(leaks) / len(FORBIDDEN_TERMS))
    return round(max(score, 0), 3), leaks


def value_fidelity_score(report_text, row):
    required_numbers = get_required_numbers(row)
    prob_normal = row["prob_normal"] * 100
    prob_mci = row["prob_mci"] * 100
    required_numbers = required_numbers + [f"{prob_normal:.1f}", f"{prob_mci:.1f}"]

    found = sum(1 for num in required_numbers if num in report_text)
    return found / len(required_numbers)


def dominant_drawing_score(report_text, row):
    attn = {
        "Clock Drawing": row["attn_pool_clock"],
        "Copy Drawing": row["attn_pool_copy"],
        "Trail Making": row["attn_pool_trail"],
    }
    hot = {
        "Clock Drawing": row["clock_hot_area_pct"],
        "Copy Drawing": row["copy_hot_area_pct"],
        "Trail Making": row["trail_hot_area_pct"],
    }
    dominant_attn_task = max(attn, key=attn.get)
    dominant_hot_task = max(hot, key=hot.get)

    checks = 2
    hits = 0
    if dominant_attn_task in report_text:
        hits += 1
    if dominant_hot_task in report_text:
        hits += 1
    return hits / checks


def conclusion_consistency_score(report_text, row):
    prob_mci = row["prob_mci"] * 100
    expected = (
        "suggestive of mild cognitive impairment (mci)."
        if prob_mci >= 50
        else "not suggestive of mild cognitive impairment (mci)."
    )
    return 1.0 if expected in report_text.lower() else 0.0


def disclaimer_present_score(report_text):
    # FIX: normalize typographic quotes/dashes before comparing against the
    # straight-ASCII DISCLAIMER_TEXT reference. Without this, a model-emitted
    # curly apostrophe in "patient's" (e.g. from Qwen2.5) silently fails an
    # otherwise-correct disclaimer, even after enforce_disclaimer() has run,
    # since enforcement happens on report_text but this function previously
    # did a raw, non-normalized comparison.
    normalized = normalize_quotes(report_text.lower())
    return 1.0 if DISCLAIMER_TEXT in normalized else 0.0


def cross_report_similarity_score(all_reports):
    if len(all_reports) < 2:
        return None

    def normalize(text):
        text = re.sub(r"##.*", "", text)
        text = re.sub(r"\d+(\.\d+)?%?", "", text)
        text = re.sub(r"[^a-z\s]", " ", text.lower())
        return set(text.split())

    token_sets = [normalize(r) for r in all_reports]
    similarities = []
    for i in range(len(token_sets)):
        for j in range(i + 1, len(token_sets)):
            a, b = token_sets[i], token_sets[j]
            if not a or not b:
                continue
            overlap = len(a & b) / len(a | b)
            similarities.append(overlap)

    return round(sum(similarities) / len(similarities), 3) if similarities else None


def evaluate_report(report_text, row):
    comp = completeness_score(report_text)
    term_score, leaked_terms = terminology_leak_score(report_text)
    val_fid = value_fidelity_score(report_text, row)
    dom_score = dominant_drawing_score(report_text, row)
    concl_score = conclusion_consistency_score(report_text, row)
    disclaimer_score = disclaimer_present_score(report_text)

    final = (
        0.20 * comp
        + 0.20 * term_score
        + 0.25 * val_fid
        + 0.15 * dom_score
        + 0.15 * concl_score
        + 0.05 * disclaimer_score
    )

    return {
        "completeness": round(comp, 3),
        "terminology_cleanliness": term_score,
        "leaked_terms": ", ".join(leaked_terms) if leaked_terms else "none",
        "value_fidelity": round(val_fid, 3),
        "dominant_drawing_accuracy": round(dom_score, 3),
        "conclusion_consistency": round(concl_score, 3),
        "disclaimer_present": round(disclaimer_score, 3),
        "final_score_0_100": round(final * 100, 2),
    }


# ==========================================================
# LLM-JUDGE EVALUATION MODULE
# ==========================================================
# The heuristic scores above are fast, deterministic, and great for the
# "hard" checks (did every required number appear, is the disclaimer exact,
# etc). But they can't judge things like clinical readability or whether the
# prose plausibly hallucinated something not in the input. For that, we use
# the model itself as a second-pass judge: it's shown the verified findings
# fed to the writer, plus the *finished* report, and asked to score it on a
# 1-5 scale across several dimensions, returned strictly as JSON.

JUDGE_SYSTEM_PROMPT = """
You are a strict clinical-documentation quality auditor.

You will be given two things:
1. The verified structured findings that were supplied to a report-writing
   assistant.
2. The finished report that assistant produced from those findings.

Score the report ONLY on how faithfully and professionally it reflects the
supplied findings. Do NOT judge whether the underlying clinical conclusion
is medically correct in some absolute sense -- only judge whether the report
is faithful to what it was given, well-structured, and free of invented
content.

Score each dimension on an integer scale from 1 (very poor) to 5 (excellent):

- clinical_correctness: does the report use cautious, non-diagnostic,
  clinically appropriate language consistent with the supplied findings?
- completeness: are all four required sections present and does each cover
  what it should (drawing assessment, clinical findings, clinical
  impression, recommendation)?
- readability: is the report clear, well-organized, natural clinical prose?
- professional_medical_style: does it read like something a neurologist
  would actually write, in tone and vocabulary?
- consistency_with_findings: do the report's statements match the supplied
  numeric findings and conclusion, without contradiction?
- hallucination_absence: 5 means the report contains NO facts, values,
  diagnoses, or claims beyond what was supplied; 1 means it invents
  significant unsupported content.
- overall: your holistic 1-5 judgment of report quality.

Return ONLY a single JSON object and nothing else -- no markdown fences, no
explanation, no extra text before or after it. Example format:

{"clinical_correctness": 5, "completeness": 5, "readability": 4, "professional_medical_style": 5, "consistency_with_findings": 5, "hallucination_absence": 5, "overall": 5}
"""

JUDGE_KEYS = [
    "clinical_correctness",
    "completeness",
    "readability",
    "professional_medical_style",
    "consistency_with_findings",
    "hallucination_absence",
    "overall",
]


def build_judge_user_prompt(row, report_text):
    findings_prompt = build_user_prompt(row, correction_note="", style=None)
    return f"""
Here are the verified findings that were supplied to the report-writing
assistant:

{findings_prompt}

=========================================================
FINISHED REPORT TO AUDIT
=========================================================

{report_text}

=========================================================

Score this report now. Return ONLY the JSON object.
"""


def parse_judge_json(raw_text):
    text = raw_text.strip()
    text = re.sub(r"^```(json)?", "", text.strip(), flags=re.IGNORECASE).strip()
    text = re.sub(r"```$", "", text.strip()).strip()
    match = re.search(r"\{.*\}", text, flags=re.DOTALL)
    if not match:
        return None
    try:
        return json.loads(match.group(0))
    except (json.JSONDecodeError, ValueError):
        return None


def llm_judge_score(model, tokenizer, row, report_text, patient_id, max_new_tokens=250):
    """Runs the model a second time as an auditor and returns a dict of
    1-5 integer scores (or None for any dimension it failed to produce)."""
    user_prompt = build_judge_user_prompt(row, report_text)

    raw = generate(
        model=model,
        tokenizer=tokenizer,
        system_prompt=JUDGE_SYSTEM_PROMPT,
        user_prompt=user_prompt,
        patient_id=f"{patient_id}_judge",
        attempt=0,
        max_new_tokens=max_new_tokens,
        temperature=0.2,  # low temperature: we want a consistent, careful judge
    )

    parsed = parse_judge_json(raw)
    if parsed is None:
        print(f"WARNING: Patient {patient_id} — LLM judge output could not be "
              f"parsed as JSON. Raw output: {raw[:200]!r}")
        return {k: None for k in JUDGE_KEYS}

    result = {}
    for k in JUDGE_KEYS:
        v = parsed.get(k)
        try:
            v = int(round(float(v)))
            v = min(5, max(1, v))
        except (TypeError, ValueError):
            v = None
        result[k] = v
    return result


def run_llm_judge_evaluation(outdir, df, reports_dict, model, tokenizer,
                              checkpoint_every=10):
    """Runs the LLM-judge pass with progress logging, periodic checkpoint
    saving, and per-patient error isolation. On a single Colab GPU, running
    ~138 extra generations back-to-back after already generating (and often
    regenerating) 138 reports is a common place to hit a CUDA OOM or a long
    stall -- without checkpointing, a crash here loses every judge score,
    even for patients that succeeded. This version writes progress to disk
    every `checkpoint_every` patients and never lets a single bad generation
    kill the whole pass."""
    print(f"\nRunning LLM-judge evaluation pass (second model call per report) "
          f"over {len(df)} patients...")
    judge_path = Path(outdir) / "llm_judge_evaluation.csv"
    records = []

    for i, (_, row) in enumerate(df.iterrows(), start=1):
        pid = row["patient_id"]
        report_text = reports_dict.get(str(pid), "")
        if not report_text:
            print(f"  [{i}/{len(df)}] Patient {pid}: no report found, skipping.")
            continue

        try:
            scores = llm_judge_score(model, tokenizer, row, report_text, pid)
        except torch.cuda.OutOfMemoryError:
            print(f"  [{i}/{len(df)}] Patient {pid}: CUDA OOM during judging — "
                  f"clearing cache and skipping this patient.")
            scores = {k: None for k in JUDGE_KEYS}
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
        except Exception as e:
            print(f"  [{i}/{len(df)}] Patient {pid}: judge call failed "
                  f"({type(e).__name__}: {e}) — skipping this patient.")
            scores = {k: None for k in JUDGE_KEYS}

        scores["Patient_ID"] = pid
        records.append(scores)

        if i % 5 == 0 or i == len(df):
            print(f"  [{i}/{len(df)}] judged.")

        if i % checkpoint_every == 0:
            pd.DataFrame(records).to_csv(judge_path, index=False)
            if torch.cuda.is_available():
                torch.cuda.empty_cache()

    judge_df = pd.DataFrame(records)
    judge_df.to_csv(judge_path, index=False)

    print("\n" + "=" * 70)
    print("LLM-JUDGE EVALUATION SUMMARY (mean across all patients, 1-5 scale)")
    print("=" * 70)
    numeric_cols = [c for c in JUDGE_KEYS if c in judge_df.columns]
    print(judge_df[numeric_cols].mean(numeric_only=True))
    print(f"\nSaved: {judge_path}")

    return judge_df


def run_automated_evaluation(outdir, df, reports_dict, judge_df=None):
    eval_records = []
    for _, row in df.iterrows():
        pid_str = str(row["patient_id"])
        report_text = reports_dict.get(pid_str, "")
        scores = evaluate_report(report_text, row)
        scores["Patient_ID"] = row["patient_id"]
        eval_records.append(scores)

    eval_df = pd.DataFrame(eval_records)
    eval_path = Path(outdir) / "automated_evaluation.csv"
    eval_df.to_csv(eval_path, index=False)

    similarity = cross_report_similarity_score(list(reports_dict.values()))

    print("\n" + "=" * 70)
    print("AUTOMATED (HEURISTIC) EVALUATION SUMMARY (mean across all patients)")
    print("=" * 70)
    print(eval_df[[
        "completeness", "terminology_cleanliness", "value_fidelity",
        "dominant_drawing_accuracy", "conclusion_consistency",
        "disclaimer_present", "final_score_0_100"
    ]].mean())

    if similarity is not None:
        print(f"\nCross-report prose similarity (0=fully unique, 1=identical wording): {similarity}")
        if similarity > 0.6:
            print("WARNING: reports still look templated. Consider raising temperature further")
            print("or reducing/removing more example sentences from the system prompt.")

    print(f"\nSaved: {eval_path}")

    if judge_df is not None and not judge_df.empty:
        combined = eval_df.merge(judge_df, on="Patient_ID", suffixes=("", "_llm_judge"))
        combined_path = Path(outdir) / "full_evaluation.csv"
        combined.to_csv(combined_path, index=False)
        print(f"Saved combined heuristic + LLM-judge evaluation: {combined_path}")

    return eval_df


# ==========================================================
# MAIN PIPELINE
# ==========================================================

def run(csv_path: str, outdir: str, model_name: str, use_4bit: bool, limit=None,
        run_llm_judge: bool = RUN_LLM_JUDGE):

    mount_drive_if_needed()
    csv_path = find_csv_if_missing(csv_path)

    outdir = Path(outdir)
    outdir.mkdir(parents=True, exist_ok=True)

    df = pd.read_csv(csv_path, dtype={"patient_id": str})
    df.columns = [c.strip() for c in df.columns]

    required_cols = [
        "patient_id", "prob_normal", "prob_mci",
        "attn_pool_clock", "attn_pool_copy", "attn_pool_trail",
        "clock_hot_area_pct", "copy_hot_area_pct", "trail_hot_area_pct",
    ]
    missing_cols = [c for c in required_cols if c not in df.columns]
    if missing_cols:
        raise ValueError(
            f"gradcam_summary.csv is missing expected column(s): {missing_cols}. "
            f"This script expects the CSV produced by the EfficientNet-B3 + "
            f"Transformer Grad-CAM script (attn_pool_* columns). If you're "
            f"pointing this at a CSV from a different model version, the "
            f"column names in build_user_prompt()/get_required_numbers()/"
            f"dominant_drawing_score() will need to match instead."
        )

    if limit is not None:
        df = df.head(limit)

    print("=" * 70)
    print(f"Patients found : {len(df)}")
    print("=" * 70)

    # Diagnostic: shows immediately whether "every report says not-MCI" is a
    # data issue (all rows have prob_mci < 50%) rather than an LLM issue.
    print_conclusion_distribution(df)

    create_human_evaluation_template(outdir, df)

    model, tokenizer = load_model(model_name=model_name, use_4bit=use_4bit)

    reports_dict = {}

    for idx, row in df.iterrows():
        pid = sanitize_filename(row["patient_id"])
        print(f"\n[{idx+1}/{len(df)}] Processing Patient {pid}")

        style = pick_style(row["patient_id"])

        try:
            report, still_missing = generate_report_with_retry(
                model=model,
                tokenizer=tokenizer,
                row=row,
                patient_id=row["patient_id"],
                style=style,
            )
        except torch.cuda.OutOfMemoryError:
            print(f"  -> CUDA OOM generating Patient {pid}. Clearing cache and "
                  f"skipping this patient — rerun with --limit or a smaller "
                  f"batch to pick up stragglers.")
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
            continue
        except Exception as e:
            print(f"  -> Generation failed for Patient {pid} "
                  f"({type(e).__name__}: {e}). Skipping this patient.")
            continue

        # ---- SAFETY NET: force-correct the Clinical Impression section ----
        report = enforce_clinical_impression(report, row)
        validate_conclusion_match(report, row, pid)

        # ---- SAFETY NET: force-correct the final disclaimer sentence ----
        report = enforce_disclaimer(report)
        validate_disclaimer_match(report, pid)

        if still_missing:
            print(f"  -> NOTE: Patient {pid} report saved with missing values "
                  f"{still_missing} despite retry. Flag for manual review.")
        # ---------------------------------------------------------------

        reports_dict[str(row["patient_id"])] = report

        report_path = outdir / f"{pid}_report.md"
        with open(report_path, "w", encoding="utf-8") as f:
            f.write("# Cognitive Drawing Assessment Report\n\n")
            f.write(f"**Patient ID:** {row['patient_id']}\n\n")
            f.write(report)
            f.write("\n\n---\n\n")
            f.write("## Reviewed Drawings\n\n")

            if "clock_overlay_path" in row:
                f.write(f"- Clock Drawing Review: `{Path(str(row['clock_overlay_path'])).name}`\n")
            if "copy_overlay_path" in row:
                f.write(f"- Copy Drawing Review: `{Path(str(row['copy_overlay_path'])).name}`\n")
            if "trail_overlay_path" in row:
                f.write(f"- Trail Making Review: `{Path(str(row['trail_overlay_path'])).name}`\n")

        print(f"Saved: {report_path}")

    judge_df = None
    if run_llm_judge:
        judge_df = run_llm_judge_evaluation(outdir, df, reports_dict, model, tokenizer)

    run_automated_evaluation(outdir, df, reports_dict, judge_df=judge_df)

    print("\n" + "=" * 70)
    print("All reports generated successfully.")
    print(f"Output folder:\n{outdir.resolve()}")
    print("=" * 70)


# ==========================================================
# MAIN
# ==========================================================

def main():
    parser = argparse.ArgumentParser(
        description="Generate neurologist-style MCI reports using Qwen2.5-7B-Instruct."
    )
    parser.add_argument("--csv", type=str, default=CSV_PATH)
    parser.add_argument("--outdir", type=str, default=OUTDIR)
    parser.add_argument("--model", type=str, default=MODEL_NAME)
    parser.add_argument("--limit", type=int, default=LIMIT)
    parser.add_argument("--no4bit", action="store_true")
    parser.add_argument("--no-llm-judge", action="store_true",
                         help="Skip the LLM-judge evaluation pass.")

    if running_in_notebook():
        class Args:
            pass
        args = Args()
        args.csv = CSV_PATH
        args.outdir = OUTDIR
        args.model = MODEL_NAME
        args.limit = LIMIT
        args.no4bit = not USE_4BIT
        args.no_llm_judge = not RUN_LLM_JUDGE
    else:
        args = parser.parse_args()

    run(
        csv_path=args.csv,
        outdir=args.outdir,
        model_name=args.model,
        use_4bit=not args.no4bit,
        limit=args.limit,
        run_llm_judge=not args.no_llm_judge,
    )


if __name__ == "__main__":
    main()

Found:
/content/drive/MyDrive/MCI_Results_efficientnet_b3_TransformerConcatFusion/gradcam/gradcam_summary.csv
Patients found : 138
CONCLUSION DISTRIBUTION (derived directly from CSV prob_mci column)
  Suggestive of MCI      : 4
  Not suggestive of MCI  : 134

Human evaluation template saved:
/content/drive/MyDrive/MCI_Results_efficientnet_b3_TransformerConcatFusion/gradcam/reports1/human_evaluation_template.csv

Loading Qwen/Qwen2.5-7B-Instruct ...


config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

...

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

Model loaded successfully.

Confirmed do_sample: True
Confirmed temperature: 0.85

[1/138] Processing Patient 447807127603920815
  -> Attempt 1 for Patient 447807127603920815 missing values ['2.7', '3.5']. Retrying with correction note...
  -> Retry succeeded on attempt 2 for Patient 447807127603920815.
Saved: /content/drive/MyDrive/MCI_Results_efficientnet_b3_TransformerConcatFusion/gradcam/reports1/447807127603920815_report.md

[2/138] Processing Patient 4335393936854271175
  -> Attempt 1 for Patient 4335393936854271175 missing values ['5.9', '10.4']. Retrying with correction note...
  -> Patient 4335393936854271175: still missing ['8.7'] after 2 attempts. Using best available attempt.
  -> NOTE: Patient 4335393936854271175 report saved with missing values ['8.7'] despite retry. Flag for manual review.
Saved: /content/drive/MyDrive/MCI_Results_efficientnet_b3_TransformerConcatFusion/gradcam/reports1/4335393936854271175_report.md

[3/138] Processing Patient 5944318648660480123
Saved: 